# Example experiment on the microscope-agnostic layer

The general flow a smart-microscopy workflow follows on top of the layer. The
workflow talks only to the layer - it never imports a vendor driver.

1. connect
2. select the reactivatable states for each scan mode (prescan, target)
3. get the positions to visit
4. acquire: set a state, then run every position (set_xyz, acquire, save)

## Setup

Register the bundled mock driver so the flow runs with no hardware. Run this
notebook with its own folder as the working directory.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()  # .../microscopes/microscope_agnostic_layer
sys.path.insert(0, str(here.parents[0]))  # microscopes/ source root
sys.path.insert(0, str(here / "tests"))  # the mock driver

import mock_driver  # noqa: E402

mock_driver.register_mock()

## Connect

Select the driver and open the session, then show the capability menu (the
options the driver advertises and which one is active).

In [ ]:
from microscope_agnostic_layer import connect

mic = connect(vendor="mock")

In [ ]:
mic.capabilities

## Select states

Capture a reactivatable state per scan mode. States are opaque to the layer -
only the driver interprets them. One cell per state.

In [ ]:
prescan = mic.get_state()
prescan["mutable"]["laser_power"] = 2.0
prescan

In [ ]:
target = mic.get_state()
target["mutable"]["laser_power"] = 8.0
target["mutable"]["gain"] = 2.0
target

## Get the positions

In [ ]:
positions = mic.get_initial_positions()
positions

## Acquisition

Set the mode, then run every position: set_xyz, acquire, save. Re-run with
`mic.set_state(target)` for the target scan.

In [ ]:
mic.set_state(prescan)

In [ ]:
for i, pos in enumerate(positions):
    mic.set_xyz(pos["x"], pos["y"], pos["z"])
    mic.acquire(backlash_correction=True)
    print(mic.save(name=f"prescan_{i:03d}", position=pos))

## Done

In [ ]:
mic.disconnect()